# Anmeldungsverlauf öffentlicher Schulungen

Dieses Notebook zeigt, wie sich die Teilnehmerzahl öffentlicher Schulungen über
die Zeit entwickelt hat - unabhängig von der Umsatzprognose des
Hauptdashboards (`01_dashboard.ipynb`). Es liest die Daten nur; es verändert
nichts.

**So wird es benutzt:** oben im Menü *Laufzeit → Alle ausführen*, dann von oben
nach unten lesen. Alle Diagramme sind interaktiv – mit dem Mauszeiger über einem
Balken stehen die genauen Zahlen.

In [ ]:
# @title Umgebung einrichten und Anmeldungsverlauf laden

import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"
    SETUP_URL = (
        "https://raw.githubusercontent.com/it-agile/umsatzprognose-clockodo/"
        f"{PAKET_REF}/notebooks/setup.py"
    )
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
    !curl -sL "$SETUP_URL" -o setup.py

import setup

ab_jahr = 2022

verlauf = setup.anmeldungsverlauf(ab_jahr=ab_jahr)

## Anmeldungen je Monat

Teilnehmerzahl öffentlicher Schulungen je Monat, insgesamt (aus der Spalte `TN Zahl`
derselben Tabelle, aus der auch der Umsatz für die Umsatzprognose stammt), dazu eine
lineare Trendlinie - wie in der internen ZDF-Präsentation.

Der Betrachtungszeitraum (`monate_fenster`) ist unten änderbar.

In [ ]:
# @title Anmeldungen je Monat

from datetime import date

from umsatzprognose.darstellung import diagramme

monate_fenster = 13
stichtag = date.today()

verlauf_fenster = verlauf.letzte(monate=monate_fenster, stichtag=stichtag)

diagramme.anmeldungsverlauf(verlauf_fenster)

## Anmeldungen je Kategorie

Bei Bedarf ein Blick je Kategorie und Monat statt nur der Gesamtzahl - eine Kategorie
je Zeile, ein Monat je Spalte, mit einer Summenspalte je Kategorie und einer
abschließenden Gesamt-Zeile je Monat. Welche Schulungstypen zu welcher Kategorie
zählen, ist unten frei konfigurierbar - nicht gelistete Typen zählen automatisch zu
"Sonstige".

In [ ]:
# @title Kategorien konfigurieren

from umsatzprognose.schulungen import kategorien_automatisch

# Schulungstyp wie in der Spalte "Schulung" des Sheets, siehe verlauf.schulungstypen.
# Alles hier nicht Gelistete zählt zu "Sonstige". Aus SCHULUNGEN_KATEGORIEN gelesen
# (Colab-Secrets in Colab, sonst .env) statt hier als Konstante gepflegt - dieselbe
# Quelle wie die Webapp, keine zwei unabhängig gepflegten Kopien mehr.
KATEGORIEN = kategorien_automatisch()

In [ ]:
# @title Anmeldungen je Kategorie

from umsatzprognose.darstellung import tabellen

unkategorisiert = sorted(
    set(verlauf_fenster.schulungstypen) - {typ for typen in KATEGORIEN.values() for typ in typen}
)
if unkategorisiert:
    print("Nicht kategorisiert (zählt zu 'Sonstige'):")
    for typ in unkategorisiert:
        print(f"- {typ}")

tabellen.anmeldungstabelle(verlauf_fenster, KATEGORIEN)

## Was zu den Zahlen zu wissen ist

Ein fehlendes oder nicht lesbares Jahr wird hier gemeldet, statt die Auswertung
scheitern zu lassen.

In [ ]:
# @title Hinweise zu den Zahlen

if verlauf.abbildungshinweise:
    for hinweis in verlauf.abbildungshinweise:
        print(f"- {hinweis.text}")
else:
    print("Keine Hinweise.")